# Pyspark intro
 
Pyspark is the python API to Apache Spark 

A short note on notebooks in databricks
- SQL notebook
- Python notebook 

The choice makes all cells defailt to that choice eg. SQL -> cells become SQL by default 


In [0]:

DATA_PATH = "/Volumes/data/olympic_games/raw_data/"

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header = True)
df_athletes


In [0]:
df_athletes.limit(2).display()

In [0]:
df_athletes.printSchema()

In [0]:
(df_athletes.count(), len(df_athletes.columns))

## Infer the schema

We note that age, height becomes strings, that is because it has null values 

In [0]:
df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header = True, inferSchema=True)
df_athletes


In [0]:
display(df_athletes)

### Define explicit schema

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema =spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header = True, schema=schema)
display(df_athletes_schema)

# EDA

## PySpark transformations

In [0]:
df_athletes_schema.groupBy("NOC", "Medal").count().filter(
    "NOC IN ('SWE', 'FIN', 'NOR', 'DEN', 'ISL') AND Medal != 'NA'"
).sort("NOC", "Medal").display()

### Spark SQL 

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

spark.sql(
    "SELECT * FROM df_athletes_schema WHERE NOC = 'SWE' AND Sport = 'Table Tennis' AND Medal != 'NA'"
).display()

# To do pick out swedish medals in table tennis

## Count medals and plot

In [0]:
df_swe_medals = spark.sql("""
          SELECT 
          sport, COUNT(medal) AS medal_count
          FROM df_athletes_schema
          WHERE NOC = 'SWE' AND medal IN ('Bronze', 'Silver', 'Gold')
          GROUP BY sport 
          ORDER BY medal_count DESC
          """)

df_swe_medals.display()

In [0]:
fig = df_swe_medals.plot(kind="bar", y="sport", x="medal_count")
fig.update_layout(
    autosize=False,
    width=1000,
    height=500,
    title="Swedish Athletes in Table Tennis",
    xaxis_title="Medal Count",
    yaxis_title="Sport",
)
fig.show()

### Ingesting data to unity catalog


In [0]:
%sql
SHOW SCHEMAS IN data;

CREATE TABLE IF NOT EXISTS 
data.olympic_games.sweden_medals AS 
(
    SELECT 
        name, 
        age, 
        height, 
        weight, 
        year, 
        sport,
        medal 
    FROM df_athletes_schema
    WHERE 
        NOC = 'SWE' AND 
        medal in ('Gold', 'Silver','Bronze')
)

In [0]:
%sql
SELECT name, sport, age
FROM data.olympic_games.sweden_medals
WHERE sport = 'Swimming' and age > 24
